In [1]:
import pymupdf

#opens a pdf file
doc = pymupdf.open("git-cheat-sheet-education.pdf") 

In [4]:
# opening remote files

import pymupdf

import requests

r = requests.get("https://education.github.com/git-cheat-sheet-education.pdf")
data = r.content
doc = pymupdf.Document(stream = data)

In [11]:
# opening files as text

doc = pymupdf.open("ankit.py", filetype = "txt")

Extracting text from a pdf

In [2]:
import pymupdf

doc = pymupdf.open("git-cheat-sheet-education.pdf")
out = open("output.txt", "wb" )

for page in doc: # iteratring over pages of doc
    text = page.get_text().encode("utf8") #getting text of a page and encoding using utf8, storing in text
    out.write(text) # writing or inserting text into output txt file
    out.write(bytes((12,))) # same as out.write(b"\f") which is a page breaker in utf8 says that page ended
out.close()

Note: if document contain image based text content the use OCR on the page for subsequent text extraction:

tp = page.get_textpage_ocr();
text = page.get_text(textpage=tp)

In [12]:
# extracting table from a page

"""Table can be found and extracted from any document"""

import pymupdf
from pprint import pprint

doc = pymupdf.open("git-cheat-sheet-education.pdf") # open document
page = doc[0] # get the first page of pdf
tabs = page.find_tables() # locate and extract any table on page
print(f"{len(tabs.tables)} found on {page}") # pring no. of tables found

if tabs.tables: # table is found
    pprint(tabs[0].extract()) # print content of first table

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
0 found on page 0 of <git-cheat-sheet-education.pdf, doc# 8>


In [3]:
# getting All Annotations from a Document

import pymupdf

for page in doc:
    for annot in page.annots():
        print(f"Annotation on page: {page.number} with type: {annot.type} and rect: {annot.rect}")

In [4]:
# gives details about the document
print(doc.metadata)

{'format': 'PDF 1.3', 'title': 'git-cheat-sheet-education', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Adobe Illustrator CC (Macintosh)', 'producer': 'Mac OS X 10.9.1 Quartz PDFContext', 'creationDate': "D:20140224195805Z00'00'", 'modDate': "D:20140224195805Z00'00'", 'trapped': '', 'encryption': None}


In [5]:
doc.load_page(1) # read a page

page 1 of <git-cheat-sheet-education.pdf, doc# 2>

Data cleaning

In [ ]:
import re
import sys
import unicodedata
from pathlib import Path
from collections import Counter


def fix_encoding(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)


Reading: --f=/run/user/1000/jupyter/runtime/kernel-v376bde775dcea5f3b7a27ea49a90efdf925b032d4.json


FileNotFoundError: File not found: --f=/run/user/1000/jupyter/runtime/kernel-v376bde775dcea5f3b7a27ea49a90efdf925b032d4.json

In [ ]:
"""
text_cleaner.py
===============================================================
Quasar Summarizer  -  Complete Text Cleaning Pipeline
===============================================================

Cleans raw .txt content so the AI model (distilBART) receives
the highest quality input possible.

Every step uses regex and is fully documented:
  - What it fixes
  - Why it matters for summarization quality
  - Before / After example

Usage:
    python text_cleaner.py                  # built-in demo
    python text_cleaner.py input.txt        # clean a file, print result
    python text_cleaner.py in.txt out.txt   # clean and save

Import in your project:
    from text_cleaner import clean_text
    cleaned = clean_text(raw_text)

Author: Quasar
"""

import re
import sys
import unicodedata
from pathlib import Path
from collections import Counter


# ---------------------------------------------------------------
#  STEP 1  -  Fix Encoding Artifacts
# ---------------------------------------------------------------

def fix_encoding(text: str) -> str:
    """
    Fix garbled characters from wrong encoding detection (mojibake).

    WHY IT MATTERS
    When a file is read with the wrong encoding (e.g. a UTF-8 file
    opened as Latin-1), you get garbage sequences instead of proper
    characters. The AI tokenizer treats each garbage sequence as
    unknown tokens, wasting context window and confusing the model.

    HOW IT WORKS
    1. unicodedata.normalize("NFKC") converts compatibility variants
       to standard forms: ligature fi->fi, superscript 2->2, etc.
    2. Regex replaces fancy typographic characters with ASCII/standard
       equivalents that the BART tokenizer handles correctly.
    3. Removes invisible zero-width and control characters.

    BEFORE : \u201cHello\u201d  it\u2019s great \u2014 let\u2019s go
    AFTER  : "Hello"  it's great - let's go
    """

    # Unicode NFKC normalization
    # Converts: fi ligature -> fi, Roman numeral II -> II,
    # superscripts/subscripts -> plain digits, etc.
    text = unicodedata.normalize("NFKC", text)

    # Fancy single quotes -> ASCII apostrophe
    # Covers: left single quote, right single quote, backtick, acute accent
    text = re.sub(r"[\u2018\u2019\u02bc\u0060\u00b4]", "'", text)

    # Fancy double quotes -> ASCII double quote
    # Covers: left/right double quotes, angle quotes (guillemets)
    text = re.sub(r"[\u201c\u201d\u00ab\u00bb\u2039\u203a]", '"', text)

    # Em dash and en dash -> spaced hyphen
    # WHY: em dashes cause NLTK sent_tokenize to sometimes miss
    # sentence boundaries ("word--word" looks like one token)
    text = re.sub(r"[\u2013\u2014\u2015\u2212\u2010\u2011]", " - ", text)

    # Ellipsis character -> three dots
    text = text.replace("\u2026", "...")

    # Bullet-like characters -> hyphen (so they become list markers)
    text = re.sub(r"[\u2022\u2023\u25e6\u2043\u2219]", "-", text)

    # Soft hyphen (invisible line-break hint in PDFs) -> empty
    text = text.replace("\u00ad", "")

    # Zero-width characters (invisible but take token space)
    text = re.sub(r"[\u200b\u200c\u200d\u200e\u200f\ufeff]", "", text)

    # Null bytes and Unicode replacement character
    text = re.sub(r"[\x00\ufffd]", "", text)

    # Non-printable ASCII control characters (keep \n \t \r)
    text = re.sub(r"[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)

    return text


# ---------------------------------------------------------------
#  STEP 2  -  Remove Headers, Footers, and Page Numbers
# ---------------------------------------------------------------

def remove_headers_footers(text: str) -> str:
    """
    Remove repeating page numbers, headers, and footer lines.

    WHY IT MATTERS
    PDF/text extraction includes every page's header and footer.
    A 100-page book repeats the chapter title 100 times. The AI
    model sees that title as critically important and produces
    summaries that obsess over the title instead of the content.

    PATTERNS REMOVED
    - Standalone page numbers: 42  -42-  [42]  (42)
    - "Page N of M" / "p.42" / "pg 7"
    - Roman numeral page numbers on their own line
    - Pure chapter/section labels: "CHAPTER 3" with no title
    - Copyright / all rights reserved lines
    - Standalone URL lines
    - ISBN / DOI / ISSN lines

    BEFORE :
        Chapter 3: Machine Learning
        content...
        - 42 -
        Chapter 3: Machine Learning
        Page 43 of 200

    AFTER  :
        content...
        content...
    """

    # Standalone page numbers: 42  -42-  [42]  (42)
    text = re.sub(
        r"^\s*[-\[\(]?\s*\d{1,4}\s*[-\]\)]?\s*$",
        "", text, flags=re.MULTILINE
    )

    # "Page N" / "Page N of M" / "p.42" / "pg 7"
    text = re.sub(
        r"\b(page|pg|p\.?)\s*\d+(\s+of\s+\d+)?\b",
        "", text, flags=re.IGNORECASE
    )

    # Roman numeral page numbers on their own line: xiv  IV  viii
    text = re.sub(
        r"^\s*[ivxlcdmIVXLCDM]{1,8}\s*$",
        "", text, flags=re.MULTILINE
    )

    # Pure chapter/section labels with no title: "CHAPTER 3"
    text = re.sub(
        r"^\s*(chapter|section|part|unit|module)\s+[\dIVXivx]+\s*$",
        "", text, flags=re.MULTILINE | re.IGNORECASE
    )

    # Copyright and legal lines
    text = re.sub(
        r"^.*?(copyright|all rights reserved|unauthorized reproduction).*?$",
        "", text, flags=re.MULTILINE | re.IGNORECASE
    )

    # URL-only lines
    text = re.sub(r"^\s*https?://\S+\s*$", "", text, flags=re.MULTILINE)

    # ISBN / DOI / ISSN lines
    text = re.sub(
        r"^\s*(isbn|doi|issn)[:\s][\d\-X\.\/]+\s*$",
        "", text, flags=re.MULTILINE | re.IGNORECASE
    )

    return text


# ---------------------------------------------------------------
#  STEP 3  -  Fix Broken Line Wrapping
# ---------------------------------------------------------------

def fix_line_breaks(text: str) -> str:
    """
    Re-join lines broken mid-sentence by PDF or text extraction.

    WHY IT MATTERS
    Exported text wraps lines at the original page width (60-80 chars),
    splitting sentences arbitrarily. The AI model sees each wrapped
    line as a potential sentence boundary and produces fragmented
    summaries that don't connect ideas across lines.

    RULE
    Join consecutive lines if BOTH of these are true:
      1. Current line does NOT end with sentence-ending punctuation
      2. Next line does NOT look like a new paragraph (blank line,
         bullet, numbered item, ALL CAPS heading, markdown header)

    BEFORE :
        The transformer architecture revolutionized natural
        language processing by introducing the attention
        mechanism in 2017.

    AFTER  :
        The transformer architecture revolutionized natural language processing by introducing the attention mechanism in 2017.
    """

    # A line ends mid-sentence if it does NOT end with . ? ! : ; " '
    ends_mid = re.compile(r"[^.?!:;\"']\s*$")

    # A line starts a new block if it matches any of these
    new_block = re.compile(
        r"^\s*$"                          # blank line
        r"|^\s{4,}"                       # heavily indented (code)
        r"|^\s*\d+[\.\)]\s+"             # numbered list: "1. " or "1) "
        r"|^\s*[-\u2022\u25e6\u2023]\s+" # bullet point
        r"|^\s*[A-Z][A-Z\s]{4,}$"        # ALL CAPS heading
        r"|^\s*#{1,6}\s"                  # markdown heading
    )

    lines = text.split("\n")
    result = []
    i = 0

    while i < len(lines):
        line = lines[i]
        # Keep joining while current line ends mid-sentence and
        # the next line is a continuation (not a new block)
        while (
            i + 1 < len(lines)
            and ends_mid.search(line)
            and not new_block.match(lines[i + 1])
            and len(lines[i + 1].strip()) > 0
        ):
            i += 1
            line = line.rstrip() + " " + lines[i].lstrip()
        result.append(line)
        i += 1

    return "\n".join(result)


# ---------------------------------------------------------------
#  STEP 4  -  Remove Symbols and Noise
# ---------------------------------------------------------------

def remove_noise(text: str) -> str:
    """
    Remove characters and patterns that carry no semantic meaning.

    WHY IT MATTERS
    Symbols like ========= or *** or HTML tags waste tokens and
    confuse the model. Numeric citation brackets [1] interrupt the
    flow of sentences and break tokenization.

    PATTERNS REMOVED
    - Decorative divider lines: ======= ------- ************
    - Repeated punctuation: !!! ...... ------
    - Markdown bold/italic: **word**  __word__  *word*
    - Markdown headers: ## Title -> Title
    - HTML tags and entities: <b>text</b>  &nbsp;  &amp;
    - Numeric citations: word [1]  [12,13]
    - Inline URLs inside sentences
    - Email addresses
    - Pipe characters (PDF table artifacts)

    BEFORE : ====== INTRO ======
             **key point** see [1] &nbsp; for <b>details</b>
    AFTER  : INTRO
             key point see   for details
    """

    # Decorative divider lines: ===  ---  ***  ___  ~~~
    text = re.sub(r"^[\s\-=_*#~+\.]{4,}\s*$", "", text, flags=re.MULTILINE)

    # Repeated identical non-word chars (3+ in a row) -> single char
    text = re.sub(r"([^\w\s])\1{2,}", r"\1", text)

    # Markdown bold/italic: **word** -> word, __word__ -> word
    text = re.sub(r"\*{1,2}(.+?)\*{1,2}", r"\1", text)
    text = re.sub(r"_{1,2}(.+?)_{1,2}",   r"\1", text)

    # Markdown headers: ## Title -> Title
    text = re.sub(r"^#{1,6}\s+", "", text, flags=re.MULTILINE)

    # HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # HTML entities
    html_entities = {
        "&nbsp;": " ", "&amp;": "&", "&lt;": "<",
        "&gt;": ">",   "&quot;": '"', "&apos;": "'",
    }
    for entity, char in html_entities.items():
        text = text.replace(entity, char)
    # Any remaining HTML entities
    text = re.sub(r"&[a-z]{2,6};", " ", text)

    # Numeric citation brackets: [1]  [12,13]  [citation needed]
    text = re.sub(r"\[\d+(?:,\s*\d+)*\]", "", text)
    text = re.sub(r"\[citation needed\]", "", text, flags=re.IGNORECASE)

    # Inline URLs
    text = re.sub(r"https?://\S+", "", text)

    # Email addresses
    text = re.sub(r"\b[\w.+-]+@[\w-]+\.[a-z]{2,}\b", "", text, flags=re.IGNORECASE)

    # Pipe characters (PDF table artifacts)
    text = re.sub(r"\|", " ", text)

    return text


# ---------------------------------------------------------------
#  STEP 5  -  Normalize Whitespace
# ---------------------------------------------------------------

def normalize_whitespace(text: str) -> str:
    """
    Standardize all spacing: tabs, multiple spaces, multiple blank lines.

    WHY IT MATTERS
    Whitespace is invisible but consumes tokens. Five blank lines costs
    the same context space as five content words. The model also reads
    blank lines as paragraph signals - too many blank lines mislead it
    into thinking the document has more sections than it does.

    BEFORE :
        Word1    word2\t\tword3


        (3 blank lines)

        Next paragraph.

    AFTER  :
        Word1 word2 word3

        Next paragraph.
    """

    # Tabs -> space
    text = re.sub(r"\t", " ", text)

    # Non-breaking and exotic Unicode spaces -> regular space
    # Covers: no-break space, em space, en space, thin space, etc.
    text = re.sub(r"[\u00a0\u1680\u2000-\u200a\u202f\u205f\u3000]", " ", text)

    # Multiple spaces on the same line -> single space
    # [^\S\n] matches whitespace that is NOT a newline
    text = re.sub(r"[^\S\n]+", " ", text)

    # Remove trailing whitespace from every line
    text = re.sub(r" +$", "", text, flags=re.MULTILINE)

    # Remove leading whitespace from every line
    text = re.sub(r"^ +", "", text, flags=re.MULTILINE)

    # 3+ consecutive blank lines -> exactly 1 blank line
    # One blank line = one \n\n = correct paragraph separation
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Strip leading/trailing whitespace from the whole document
    text = text.strip()

    return text


# ---------------------------------------------------------------
#  STEP 6  -  Fix Punctuation
# ---------------------------------------------------------------

def fix_punctuation(text: str) -> str:
    """
    Normalize punctuation spacing and fix common punctuation errors.

    WHY IT MATTERS
    Misplaced spaces around punctuation break NLTK's sent_tokenize()
    which is used for smart chunking. If "Hello .World" is not seen
    as two sentences, chunk boundaries will be wrong and the model
    gets poor input. Correct punctuation spacing is essential.

    BEFORE : Hello ,world .This is great !i went home .
    AFTER  : Hello, world. This is great! I went home.
    """

    # Remove space BEFORE punctuation
    # "Hello , world ." -> "Hello, world."
    text = re.sub(r"\s+([,\.!?;:'\)])", r"\1", text)

    # Ensure single space AFTER sentence-ending punctuation before capital
    # "Hello.World" -> "Hello. World"
    text = re.sub(r"([.!?])([A-Z])", r"\1 \2", text)

    # Add space after comma if missing before a non-digit
    # "apples,oranges" -> "apples, oranges"  (but not "1,000")
    text = re.sub(r",([^\s\d])", r", \1", text)

    # Fix space inside brackets: "( hello )" -> "(hello)"
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    text = re.sub(r"\[\s+", "[", text)
    text = re.sub(r"\s+\]", "]", text)

    # Collapse repeated punctuation: "Really??" -> "Really?"
    text = re.sub(r"([!?]){2,}", r"\1", text)

    # Collapse 4+ dots to ellipsis: "word....." -> "word..."
    text = re.sub(r"\.{4,}", "...", text)

    # Fix lone lowercase "i" -> "I" (very common OCR error)
    # Only matches "i" as a standalone word, not part of other words
    text = re.sub(r"(?<!\w)\bi\b(?!\w)", "I", text)

    # Capitalize first letter after ". " if it is lowercase
    # "end of sentence. next sentence" -> "end of sentence. Next sentence"
    def cap_after_period(m):
        return m.group(1) + " " + m.group(2).upper()
    text = re.sub(r"(\.) ([a-z])", cap_after_period, text)

    return text


# ---------------------------------------------------------------
#  STEP 7  -  Remove Boilerplate Content
# ---------------------------------------------------------------

def remove_boilerplate(text: str) -> str:
    """
    Remove structural boilerplate that carries no informational value.

    WHY IT MATTERS
    Table of contents lines, figure captions, index entries, and
    legal disclaimers all get mixed into extracted text. The AI
    model treats them as content. Result: summaries say things like
    "The document contains Chapter 1, Chapter 2, Chapter 3" instead
    of summarizing what those chapters actually say.

    PATTERNS REMOVED
    - TOC lines: "Introduction ......... 5"
    - Confidentiality/legal disclaimers
    - Figure/Table caption lines: "Figure 3.1: Results"
    - Index entries: "machine learning, 12, 45, 78"
    - Cross-reference lines: "See also: Chapter 4"

    BEFORE :
        Table of Contents ........ 3
        Introduction ............. 5
        This document is confidential and intended solely for...
        Figure 3.1: Comparison of results

    AFTER  : (all of the above removed)
    """

    # TOC lines: "Some text ......... 12"
    text = re.sub(
        r"^.{3,60}\.{3,}\s*\d{1,4}\s*$",
        "", text, flags=re.MULTILINE
    )

    # Confidentiality / legal disclaimers
    legal_patterns = [
        r"^.*?this (document|report|message) is (confidential|proprietary).*?$",
        r"^.*?intended solely for.*?$",
        r"^.*?if you (have received|are not the intended).*?$",
        r"^.*?all rights reserved.*?$",
        r"^.*?unauthorized (use|reproduction|distribution).*?$",
        r"^.*?terms and conditions.*?$",
    ]
    for pattern in legal_patterns:
        text = re.sub(pattern, "", text, flags=re.MULTILINE | re.IGNORECASE)

    # Figure / Table / Chart caption lines
    text = re.sub(
        r"^\s*(figure|fig\.|table|chart|graph|diagram|appendix)\s+[\d\.]+[:\-]?.*?$",
        "", text, flags=re.MULTILINE | re.IGNORECASE
    )

    # Index entries: "machine learning, 12, 45, 78"
    text = re.sub(
        r"^[A-Za-z][a-zA-Z\s\-]{2,40},\s*\d+(,\s*\d+)+\s*$",
        "", text, flags=re.MULTILINE
    )

    # Cross-reference lines
    text = re.sub(
        r"^\s*(see also|refer to|see section|for more information see).*?$",
        "", text, flags=re.MULTILINE | re.IGNORECASE
    )

    return text


# ---------------------------------------------------------------
#  STEP 8  -  Normalize Abbreviations and Numbers
# ---------------------------------------------------------------

def normalize_text(text: str) -> str:
    """
    Expand abbreviations that confuse the sentence tokenizer,
    and normalize inconsistent number formatting.

    WHY IT MATTERS
    "e.g." contains a period. NLTK's sent_tokenize() sometimes treats
    it as a sentence boundary and splits your chunk there. If chunks
    are split at "e.g." instead of at real sentence boundaries, the
    model gets unbalanced input and produces worse summaries.

    Thousands separators in numbers ("1,000,000") cause the tokenizer
    to split on commas, treating "1" and "000" as separate tokens.

    BEFORE : E.g. this is a problem. i.e. it breaks chunking.
             The model has 175,000,000 parameters. Rate: 75 percent.
    AFTER  : For example this is a problem. That is it breaks chunking.
             The model has 175000000 parameters. Rate: 75%
    """

    # Expand period-containing abbreviations
    abbrevs = {
        r"\be\.g\.\s":    "for example ",
        r"\bi\.e\.\s":    "that is ",
        r"\betc\.\s":     "etcetera ",
        r"\bvs\.\s":      "versus ",
        r"\bDr\.\s":      "Dr ",
        r"\bMr\.\s":      "Mr ",
        r"\bMrs\.\s":     "Mrs ",
        r"\bMs\.\s":      "Ms ",
        r"\bProf\.\s":    "Prof ",
        r"\bSt\.\s":      "St ",
        r"\bAve\.\s":     "Ave ",
        r"\bapprox\.\s":  "approximately ",
        r"\bref\.\s":     "reference ",
        r"\bfig\.\s":     "figure ",
        r"\beq\.\s":      "equation ",
        r"\bvol\.\s":     "volume ",
        r"\bno\.\s":      "number ",
        r"\bp\.\s":       "page ",
        r"\bpp\.\s":      "pages ",
    }
    for pattern, replacement in abbrevs.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    # Remove thousands separators: 1,000,000 -> 1000000
    text = re.sub(r"(\d),(\d{3})", r"\1\2", text)

    # Normalize percentage spacing: "75 %" -> "75%"
    text = re.sub(r"(\d)\s+%", r"\1%", text)
    text = re.sub(r"(\d)\s+percent\b", r"\1%", text, flags=re.IGNORECASE)

    return text


# ---------------------------------------------------------------
#  STEP 9  -  Quality Validation
# ---------------------------------------------------------------

def validate(text: str) -> dict:
    """
    Run quality checks on cleaned text. Returns a diagnostic report.

    Checks:
    - Word count (too short = bad summary)
    - Estimated BART token count (>900 = must chunk before summarizing)
    - Average sentence length (too short = fragmented/non-prose text)
    - Repeated lines (leftover headers/footers)

    Returns dict with keys:
        ok               - True if safe to send to summarizer
        word_count       - total words
        char_count       - total characters
        sentence_count   - estimated sentence count
        estimated_tokens - approx BART tokens (word_count / 0.75)
        needs_chunking   - True if > 900 estimated tokens
        warnings         - list of warning strings
    """
    warnings = []

    word_count  = len(text.split())
    char_count  = len(text)
    sentences   = re.split(r"(?<=[.!?])\s+", text.strip())
    sent_count  = len([s for s in sentences if len(s.split()) > 3])
    # 1 BART token ~ 0.75 English words
    est_tokens  = int(word_count / 0.75)

    if word_count < 10:
        warnings.append(
            "CRITICAL: text has fewer than 10 words. Cannot summarize."
        )
    elif word_count < 50:
        warnings.append(
            "Text is very short (<50 words). Summary quality will be low."
        )

    avg_sent = word_count / max(sent_count, 1)
    if avg_sent < 5:
        warnings.append(
            f"Average sentence length is {avg_sent:.1f} words. "
            "Text appears fragmented (lists or table cells). "
            "Summarizer works best on continuous prose."
        )

    # Check for repeated lines (unremoved header/footer)
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    repeated = [
        (l, c) for l, c in Counter(lines).items()
        if c >= 3 and len(l) > 5
    ]
    if repeated:
        examples = ", ".join(
            f'"{l[:30]}" x{c}' for l, c in repeated[:3]
        )
        warnings.append(
            f"Repeated lines found (possible leftover header/footer): {examples}"
        )

    return {
        "ok":               word_count >= 10,
        "word_count":       word_count,
        "char_count":       char_count,
        "sentence_count":   sent_count,
        "estimated_tokens": est_tokens,
        "needs_chunking":   est_tokens > 900,
        "warnings":         warnings,
    }


# ---------------------------------------------------------------
#  MAIN PIPELINE  -  clean_text()
# ---------------------------------------------------------------

def clean_text(text: str, verbose: bool = False) -> str:
    """
    Run the complete 9-step cleaning pipeline.

    Step order matters:
      1. fix_encoding           - fix chars first so later regex works correctly
      2. remove_headers_footers - remove structural noise before fixing line breaks
      3. fix_line_breaks        - re-join splits (needs clean lines to work correctly)
      4. remove_noise           - remove symbols and HTML
      5. normalize_whitespace   - collapse gaps created by noise removal
      6. fix_punctuation        - fix spacing around punctuation
      7. remove_boilerplate     - remove TOC lines, captions, disclaimers
      8. normalize_text         - expand abbreviations, fix number formatting
      9. normalize_whitespace   - final pass to clean gaps from all steps above

    Args:
        text    : Raw input text string
        verbose : If True, prints word count after each step (useful for debugging)

    Returns:
        Cleaned string ready for the AI summarizer
    """

    steps = [
        ("Fix encoding",           fix_encoding),
        ("Remove headers/footers", remove_headers_footers),
        ("Fix line breaks",        fix_line_breaks),
        ("Remove noise",           remove_noise),
        ("Normalize whitespace",   normalize_whitespace),
        ("Fix punctuation",        fix_punctuation),
        ("Remove boilerplate",     remove_boilerplate),
        ("Normalize text",         normalize_text),
        ("Final whitespace pass",  normalize_whitespace),
    ]

    if verbose:
        print(f"\n{'─' * 54}")
        print(f"  Starting : {len(text.split()):,} words")
        print(f"{'─' * 54}")

    for name, fn in steps:
        text = fn(text)
        if verbose:
            print(f"  [{name:<26}]  {len(text.split()):>6,} words")

    if verbose:
        print(f"{'─' * 54}\n")

    return text


# ---------------------------------------------------------------
#  UTILITY  -  Read .txt with auto encoding detection
# ---------------------------------------------------------------

def read_txt(filepath: str) -> str:
    """
    Read a .txt file, auto-detecting encoding.

    Tries encodings in order:
      utf-8-sig  (UTF-8 with BOM, common on Windows)
      utf-8      (standard)
      latin-1    (Western European, common in older docs)
      cp1252     (Windows Western European)
      ascii      (plain ASCII)

    Falls back to utf-8 with errors='replace' if all fail.

    Args:
        filepath : Path to .txt file

    Returns:
        File contents as a string
    """
    path = Path(filepath)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {filepath}")

    if path.suffix.lower() not in (".txt", ".text", ".md"):
        print(f"Warning: expected .txt, got '{path.suffix}'. Proceeding.")

    for encoding in ("utf-8-sig", "utf-8", "latin-1", "cp1252", "ascii"):
        try:
            return path.read_text(encoding=encoding)
        except (UnicodeDecodeError, LookupError):
            continue

    # Absolute fallback
    return path.read_text(encoding="utf-8", errors="replace")


# ---------------------------------------------------------------
#  COMMAND-LINE INTERFACE
# ---------------------------------------------------------------

def print_report(label: str, text: str):
    r = validate(text)
    print(f"\n  {label}")
    print(f"  {'─' * 48}")
    print(f"  Words           : {r['word_count']:,}")
    print(f"  Characters      : {r['char_count']:,}")
    print(f"  Sentences       : {r['sentence_count']:,}")
    print(f"  Estimated tokens: {r['estimated_tokens']:,}")
    chunking = "YES - chunk before summarizing" if r["needs_chunking"] else "No - fits in one call"
    print(f"  Needs chunking  : {chunking}")
    if r["warnings"]:
        for w in r["warnings"]:
            print(f"  WARNING: {w}")
    else:
        print(f"  Status          : Clean - ready to summarize")


def main():
    if len(sys.argv) == 1:
        # --- Demo mode with built-in sample dirty text ---
        dirty = (
            "\u201cHello world\u201d  it\u2019s a great day \u2014 let\u2019s begin!\n\n"
            "        CHAPTER 3\n\n"
            "        Page 42 of 100\n\n"
            "The transformer architecture  revolutionized   natural\n"
            "language processing by introducing the attention\n"
            "mechanism. It was introduced in 2017.\n\n"
            "=============================\n\n"
            "See also: Chapter 4 for more details.\n\n"
            "The model has 175,000,000 parameters.  e.g. this is how it works.\n\n"
            "Figure 1.1: Transformer Architecture Overview\n\n"
            "c 2023 Publisher. All rights reserved.\n\n"
            "Table of Contents ................ 3\n"
            "Introduction ..................... 5\n"
            "**Important:** this is a &nbsp; key &amp; concept.\n"
            "word [1] shows [2,3] citation artifacts.\n"
        )

        print("\n" + "=" * 56)
        print("  QUASAR TEXT CLEANER  -  Demo Mode")
        print("=" * 56)
        print("\n-- BEFORE -----------------------------------------------")
        print(dirty)
        print_report("Before stats", dirty)

        cleaned = clean_text(dirty, verbose=True)

        print("\n-- AFTER ------------------------------------------------")
        print(cleaned)
        print_report("After stats", cleaned)
        return

    # --- File mode ---
    input_path  = sys.argv[1]
    output_path = sys.argv[2] if len(sys.argv) > 2 else None

    print(f"\nReading: {input_path}")
    raw = read_txt(input_path)
    print_report("Before cleaning", raw)

    cleaned = clean_text(raw, verbose=True)
    print_report("After cleaning", cleaned)

    if output_path:
        Path(output_path).write_text(cleaned, encoding="utf-8")
        print(f"\nSaved cleaned file to: {output_path}")
    else:
        preview_len = 2000
        print(f"\n-- Cleaned Text (first {preview_len} chars) ----------------------")
        print(cleaned[:preview_len])
        if len(cleaned) > preview_len:
            print(f"\n... [{len(cleaned) - preview_len:,} more characters]")


if __name__ == "__main__":
    main()